In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from arch import arch_model

In [2]:
tickers = ["AAPL", "MSFT", "JPM", "XOM", "JNJ", "WMT"]

data = yf.download(tickers, start= "2022-01-01", end= "2026-01-01", auto_adjust=True)


[*********************100%***********************]  5 of 6 completed


return calculation 

In [3]:
price = data["Close"]
returns = price.pct_change()
returns = returns.dropna()
print(returns.head())


Ticker          AAPL       JNJ       JPM      MSFT       WMT       XOM
Date                                                                  
2022-01-04 -0.012692 -0.002682  0.037910 -0.017147 -0.018320  0.037614
2022-01-05 -0.026600  0.006664 -0.018282 -0.038388  0.013521  0.012438
2022-01-06 -0.016693 -0.003426  0.010624 -0.007902 -0.002779  0.023520
2022-01-07  0.000988  0.013518  0.009908  0.000510  0.009546  0.008197
2022-01-10  0.000116 -0.004944  0.000957  0.000732 -0.001933 -0.005952


var calculation

In [4]:
weights = np.array([1/len(tickers)]*6)
portfolio_returns = returns.dot(weights)
print(portfolio_returns.head())

Date
2022-01-04    0.004114
2022-01-05   -0.008441
2022-01-06    0.000557
2022-01-07    0.007111
2022-01-10   -0.001837
dtype: float64


In [5]:
var_95 = portfolio_returns.quantile(0.05)
print(var_95)

-0.014385757871484986


emwa var

In [6]:
lam = 0.94
r = portfolio_returns.values
variance = np.zeros(len(r))

variance[0] = r.var()

for t in range(1,len(r)):
    variance[t] = lam*variance[t-1] + (1-lam)*r[t-1]**2

print(variance[-1])              # today's variance (σ²)
print(np.sqrt(variance[-1]))     # today's volatility (σ) — sqrt of variance
print(1.65 * np.sqrt(variance[-1]))  # today's 95% VaR (z × σ)




2.318261311487772e-05
0.004814832615457958
0.00794447381550563


In [7]:
var_ewm = portfolio_returns.ewm(alpha=1-lam, adjust=False).var()
print(var_ewm[-1])

2.3248109953662865e-05


C:\Users\prane\AppData\Local\Temp\ipykernel_29904\3810184446.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(var_ewm[-1])


In [8]:
returns_pct = portfolio_returns*100
model = arch_model(returns_pct, vol = 'garch', p=1, q=1)
result = model.fit(disp='off')
print(result.summary())

                     Constant Mean - GARCH Model Results                      
Dep. Variable:                   None   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -1232.76
Distribution:                  Normal   AIC:                           2473.52
Method:            Maximum Likelihood   BIC:                           2493.16
                                        No. Observations:                 1002
Date:                Fri, Jun 26 2026   Df Residuals:                     1001
Time:                        19:48:22   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu             0.0913  2.307e-02      3.959  7.518e-05 [4.612e-0

In [9]:
forecast = result.forecast(horizon=10)
garch_vol = np.sqrt(forecast.variance)/100
garch_var = garch_vol*1.65
print(garch_var)

                h.01      h.02      h.03      h.04      h.05     h.06  \
Date                                                                    
2025-12-31  0.009056  0.009221  0.009379  0.009531  0.009678  0.00982   

                h.07      h.08      h.09      h.10  
Date                                                
2025-12-31  0.009957  0.010089  0.010217  0.010341  


historical 4 year rolling window 

In [35]:
window = 250
breaches = 0
test_days = 0
breach_hist_returns = []

for t in range (window, len(portfolio_returns)):
    past = portfolio_returns.iloc[t-window:t]
    var_historical = past.quantile(0.05)
    actual = portfolio_returns.iloc[t]
    if actual<var_historical:
        breaches += 1
        breach_hist_returns.append(actual)
    test_days += 1

print(f"Breaches: {breaches}/{test_days} = {breaches/test_days:.2%}")


Breaches: 28/752 = 3.72%


emwa var backtest

In [36]:
window = 250
breaches = 0
test_days = 0
lam = 0.94
breach_emwa_returns = []

for t in range (window, len(portfolio_returns)):
    past = portfolio_returns.iloc[t-window:t]
    var_emwa = -1.65*(np.sqrt(past.ewm(alpha=1-lam, adjust=False).var().iloc[-1]))
    actual = portfolio_returns.iloc[t]
    if actual<var_emwa:
        breaches += 1
        breach_emwa_returns.append(actual)
    test_days +=1

print(f"Breaches ={breaches}/{test_days} = {breaches/test_days:.2%}")
    
    

Breaches =33/752 = 4.39%


garch var being re fit every 50 days

In [37]:
window = 250
breaches = 0
test_days = 0
result = None
breach_garch_returns = []

for t in range (window, len(portfolio_returns)):
    past = returns_pct.iloc[t-window:t]

    if t%50 == 0 or result is None:
        garch_model = arch_model(past, vol = 'garch', p=1, q=1)
        result = garch_model.fit(disp='off')
    forecast = result.forecast(horizon=1)
    garch_var = -1.65*(np.sqrt(forecast.variance.values[-1, 0])/100)

    actual = portfolio_returns.iloc[t]

    if actual<garch_var:
        breaches+=1
        breach_garch_returns.append(actual)
    test_days+=1

print(f"Breaches ={breaches}/{test_days} = {breaches/test_days:.2%}")


Breaches =37/752 = 4.92%


calculating es

In [39]:
historical_es = np.mean(breach_hist_returns)
emwa_es = np.mean(breach_emwa_returns)
garch_es = np.mean(breach_garch_returns)

print(f"historical ES: {historical_es}\nEMWA ES: {emwa_es}\nGARCH ES: {garch_es}")

historical ES: -0.017866590047316024
EMWA ES: -0.0164573816793095
GARCH ES: -0.016118813506245586


In [40]:
print("Kurtosis:", portfolio_returns.kurtosis())   # >0 (excess) means fatter than normal
print("Skew:", portfolio_returns.skew())            # <0 means crashes fatter than rallies

Kurtosis: 7.775862800321227
Skew: 0.2656604088454753


In [ ]:
import numpy as np
mu = portfolio_returns.mean()
sigma = portfolio_returns.std()
N = 10000
simulated = np.random.normal(mu, sigma, N)
simulated_t =
mc_var = np.percentile(simulated, 5)

print(f"Monte Carlo VaR (normal): {mc_var:.4f}")

Monte Carlo VaR (normal): -0.0150
